#TODO

- prova multipli nn su 27 modelli
- function documentation
- import all the functions
- hyperparam refined
- train full df
- predictions



# Project

In [1]:
import pandas as pd
from thefuzz import process
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

from scipy import stats
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor

import numpy as np
from sklearn.base import BaseEstimator, clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from typing import Dict

from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor

import random
from copy import deepcopy
import time

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN

In [2]:
train_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/train.csv")

In [3]:
train_df = train_df.set_index("carID")
train_df = train_df.drop(columns=["paintQuality%"])

In [4]:
train_df.head()

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage
carID,,,,,,,,,,,,
69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,4.000000,0.0
53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,1.000000,0.0
6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,4.000000,0.0
29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,-2.340306,0.0
10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,3.000000,0.0


### Making sure that the model prices are not typos

In [5]:
#df_cleaned = simple_processing(train_df)
#df_cleaned = df_cleaned[df_cleaned["model"].notna()]
#df_price_model = df_cleaned[["Brand","model", "price"]].copy()

In [6]:

"""# Get the list of unique models
models = df_price_model['model'].unique()

# Loop through each model and create a boxplot
#for model in models:
sns.catplot(data=df_price_model, y='price', col='model', kind='box', col_wrap=4, height=3, sharey=False)
plt.show()"""

"# Get the list of unique models\nmodels = df_price_model['model'].unique()\n\n# Loop through each model and create a boxplot\n#for model in models:\nsns.catplot(data=df_price_model, y='price', col='model', kind='box', col_wrap=4, height=3, sharey=False)\nplt.show()"

In [7]:

"""def is_outlier(x):
    Q1, Q3 = x.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return (x < (Q1 - 3 * IQR)) | (x > (Q3 + 3 * IQR))

# Apply the function per model group to get the outlier rows
outliers = df_price_model[df_price_model.groupby('model')['price'].transform(is_outlier)]

# Display sorted results
outliers.sort_values(['model', 'price'])
"""

"def is_outlier(x):\n    Q1, Q3 = x.quantile([0.25, 0.75])\n    IQR = Q3 - Q1\n    return (x < (Q1 - 3 * IQR)) | (x > (Q3 + 3 * IQR))\n\n# Apply the function per model group to get the outlier rows\noutliers = df_price_model[df_price_model.groupby('model')['price'].transform(is_outlier)]\n\n# Display sorted results\noutliers.sort_values(['model', 'price'])\n"

## Data cleaning

In [8]:
# TODO: Move the functions to a separate file

In [9]:
# String cleaning and Small numbers changes

def simple_processing(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don"t require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    #df["Brand"] = df["Brand"].fillna(df["Brand_mode"])  # rename new column
    #df.drop("Brand_mode", axis=1, inplace=True)  # remove the old brand column
    
    ################################################################################
    # Simple Number Cleaning
    ################################################################################

    # Cleaning numeric columns
    df["year"] = df["year"].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)

    # Round year to integer (no fractional years)
    df["year"] = df["year"].round()

    # Replace the years 2024, 2023 with NA as these values are very likley wrong (dataset is from 2020) and we want to impute them 
    df.loc[df["year"] == 2024, "year"] = pd.NA 
    df.loc[df["year"] == 2023, "year"] = pd.NA

    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df["mileage"] = abs(df["mileage"].round())
    
    # Tax: take absolute value and round
    df["tax"] = abs(df["tax"].round())
    
    # MPG: round to 1 decimal place 
    df["mpg"] = abs(df["mpg"].round(1))
    
    # Engine size: round to 1 decimal place
    df["engineSize"] = abs(df["engineSize"].round(1))
    
    # Remove paintQuality column as this is information collected by the mechanic and thus not available for pre-assesment of the car
    #df.drop("paintQuality%", axis=1, inplace=True) 
    
    # Previous owners: take absolute value and round to integer
    df["previousOwners"] = abs(df["previousOwners"].round())
    
    return df

In [10]:
def mode_imputation(df):
    """
    Get the most frequent brand for each model -> returns df with model and brand
    """
    brand_models = df.groupby("model_transformed")["Brand_transformed"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model_transformed", how="left", suffixes=("", "_mode"))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand_transformed"] = df["Brand_transformed"].fillna(df["Brand_transformed_mode"])  # rename new column
    df.drop("Brand_transformed_mode", axis=1, inplace=True)  # remove the old brand column
    

    return df

In [11]:
def remove_price_outliers(df, threshold=3):
    """
    Replaces price outliers with NaN for each model group based on the IQR method.
    
    Parameters:
    df: Input dataframe containing 'model' and 'price' columns.
    threshold: The multiplier for the IQR to define outliers. Default is 3 (extreme outliers).
    
    Returns:
    Dataframe with outliers replaced by NaN.
    """
    df_clean = df.copy()
    
    # Calculate Q1, Q3, and IQR per model
    # We use transform to broadcast the group statistics back to the original index size
    groups = df_clean.groupby('model')['price']
    q1 = groups.transform(lambda x: x.quantile(0.25))
    q3 = groups.transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1
    
    # Define bounds
    lower_bound = q1 - (threshold * iqr)
    upper_bound = q3 + (threshold * iqr)
    
    # Identify outliers
    outlier_mask = (df_clean['price'] < lower_bound) | (df_clean['price'] > upper_bound)
    
    # Replace outliers with NaN
    df_clean.loc[outlier_mask, 'price'] = np.nan
    
    return df_clean


In [12]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [13]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [14]:
# Train imputer on train

def fit_imputer(df): 
    try: 
        df = df.drop(["price"], axis=1) # Remove the price column as it would not be available for imputation and thus should not be used when training the imputer
    except KeyError: # Happens when we use testing data as we previously dropped the price column 
        pass
    
    estimator = ExtraTreesRegressor(n_estimators=50,
                                    n_jobs=-1,
                                    min_samples_leaf=10,
                                    max_depth=20,
                                    random_state=69)

    imputer = IterativeImputer(estimator=estimator,
                                max_iter=20,
                                n_nearest_features=10,
                                random_state=69,
                                initial_strategy="most_frequent")

    imputer.fit(df)
    
    return imputer

In [15]:
def apply_imputer(df, imputer):
    """
    Apply the pretrained imputer to the dataframe features only, preserving the price column.
    """
    df_imputed = df.copy()
    
    # 1. Separate price if it exists
    price_col = None
    if 'price' in df_imputed.columns:
        price_col = df_imputed['price']
        df_for_imputation = df_imputed.drop(columns=['price'])
    else:
        df_for_imputation = df_imputed

    # 2. Apply imputer to features
    imputed_values = imputer.transform(df_for_imputation)
    
    # Create a new dataframe from the imputed values to avoid dtype warnings
    # We use the same index and columns as the input to preserve structure
    df_for_imputation = pd.DataFrame(
        imputed_values, 
        columns=df_for_imputation.columns, 
        index=df_for_imputation.index
    )
    
    # 3. Re-attach price if it existed
    if price_col is not None:
        df_for_imputation['price'] = price_col
        
    # 4. Post-processing (Rounding)
    df_for_imputation[["mpg", "engineSize"]] = abs(df_for_imputation[["mpg", "engineSize"]]).round(1)
    
    int_cols = ["year", "tax", "mileage", "previousOwners", "Brand_transformed", 
                "model_transformed", "transmission_transformed", "fuelType_transformed"]
    
    existing_int_cols = [col for col in int_cols if col in df_for_imputation.columns]
    df_for_imputation[existing_int_cols] = abs(df_for_imputation[existing_int_cols]).round(0).astype(int)
    
    return df_for_imputation

In [16]:
def fit_price_imputer(df, target_col='price'):
    """
    Trains a regressor to impute missing values in the target column (price).
    
    Parameters:
    df: Dataframe containing features and the target column with NaNs.
    target_col: The name of the target column to impute.
    
    Returns:
    model: The trained regression model.
    """
    
    # Separate data into sets with known and unknown target values
    train_known = df[df[target_col].notna()]
    
    # X contains all columns except the target
    X = train_known.drop(columns=[target_col])
    y = train_known[target_col]
    
    # Initialize the estimator (same robust parameters as your previous imputer)
    estimator = ExtraTreesRegressor(n_estimators=100,
                                    n_jobs=-1,
                                    min_samples_leaf=5,
                                    max_depth=20,
                                    random_state=69)
    
    # Fit the model
    estimator.fit(X, y)
    
    return estimator

In [17]:
def apply_price_imputer(df, model, target_col='price'):
    """
    Uses the trained model to fill NaN values in the target column.
    """
    df_imputed = df.copy()
    
    # Identify rows where the target is missing
    missing_mask = df_imputed[target_col].isna()
    
    # If there are missing values, predict and fill them
    if missing_mask.sum() > 0:
        X_missing = df_imputed.loc[missing_mask].drop(columns=[target_col])
        predicted_values = model.predict(X_missing)
        df_imputed.loc[missing_mask, target_col] = predicted_values
        
    return df_imputed

In [18]:
def decode(df, encoders):
    # Iterative imputer produces ~20-30 values that are outside of the range of the encoder
    # The simplest fix is to clip does values back into the range of the encoder
 

    df["Brand_transformed"] = df["Brand_transformed"].clip(lower=0, upper=encoders["Brand"].classes_.shape[0]-1).astype(int)
    df["Brand"] = encoders["Brand"].inverse_transform(df["Brand_transformed"])

    df["transmission_transformed"] = df["transmission_transformed"].clip(lower=0, upper=encoders["transmission"].classes_.shape[0]-1).astype(int)
    df["transmission"] = encoders["transmission"].inverse_transform(df["transmission_transformed"])
        
    # Use clip with the information of the fitted encoder, .classes_.shape gives us the dimension of the labels the encoder uses [0] is the rows - 1 because we start clipping at 0
    df["model_transformed"] = df["model_transformed"].clip(lower=0, upper=encoders["model"].classes_.shape[0]-1).astype(int)
    df["model"] = encoders["model"].inverse_transform(df["model_transformed"])
    
    df["fuelType_transformed"] = df["fuelType_transformed"].clip(lower=0, upper=encoders["fuelType"].classes_.shape[0]-1).astype(int)
    df["fuelType"] = encoders["fuelType"].inverse_transform(df["fuelType_transformed"])
    

    df.drop(columns=["Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"], inplace=True)

    return df

### Workflow for Train and Validation Sets

#### Encoding

In [19]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)

# Remove extreme price outliers
df = remove_price_outliers(df)

# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end
df_encoded, encoders = fit_transform_encoding(df)

#### Creating the stratification column

In [20]:
# Create Categorical price column with 0 < 1 < 2 for the price
df_encoded["price_cat"] = pd.qcut(df_encoded["price"], 3, labels=False)

# Combine the 10 unique brand values (1-9 & NA) with the 3 unique price category values (0-2)
stratify_col = df_encoded["Brand_transformed"].astype(str) + "_" + df_encoded["price_cat"].astype(str)

#### Imputation and decoding

In [21]:
# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=69, stratify=stratify_col)
# train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42)

# Drop price_cat column from both splits as it would introduce data leakage
train_split = train_split.drop("price_cat", axis=1)
validation_split = validation_split.drop("price_cat", axis=1)

train_split = mode_imputation(train_split)
validation_split = mode_imputation(validation_split)

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputed_validation = apply_imputer(validation_split, imputer)

# Impute the missing price values created by removing extrem outliers
# In our step to remove extrem price outliers we introduced missing values in the price column. However, in our normal imputer, which we also use for imputing the test 
# Dataset we can not fix these values as this would cause the imputer to not work on the test subset, with no price feature. 
# Thus we create a seperate price imputer to fix the missing values in the testing and validation sets. This imputer is trained on train and applied to validation
# Fit price imputer
price_imputer = fit_price_imputer(imputed_train)
# Apply price imputer
imputed_train = apply_price_imputer(imputed_train, price_imputer)
imputed_validation = apply_price_imputer(imputed_validation, price_imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputed_validation, encoders)

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [22]:
train_processed.isna().sum()

year                0
mileage             0
tax                 0
mpg                 0
engineSize          0
previousOwners      0
stated_no_damage    0
price               0
Brand               0
transmission        0
model               0
fuelType            0
dtype: int64

### Workflow for Seperated Testing Dataset

In [23]:
test_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/test.csv")

In [24]:
test_df = test_df.set_index("carID")
test_df = test_df.drop(columns=["paintQuality%"])

In [25]:
test = simple_processing(test_df)

test_df_encoded, test_encoders = fit_transform_encoding(test)

#test_imputers = fit_imputer(df_encoded_no_price, fast=False) # we use the full encoded training dataframe to train the imputers

#df_encoded_no_price = test_df_imputed.drop(["price", "price_cat"], axis=1)

# Apply imputer trained on testing data
test_df_imputed = apply_imputer(test_df_encoded, imputer)

test_processed = decode(test_df_imputed, test_encoders)


In [26]:
train_val_models = set(pd.concat([train_processed, validation_processed])["model"].unique())
test_models = set(test_processed["model"].unique())

missing_models = train_val_models - test_models


In [27]:
list(missing_models)


['caddy maxi',
 'accent',
 'kadjar',
 '230',
 'verso-s',
 'ranger',
 'urban cruiser',
 '220',
 '200',
 'getz',
 'escort',
 'streetka',
 's8']

## Feature Engineering

In [28]:
train_processed.index = train_processed.index.astype(int)
validation_processed.index = validation_processed.index.astype(int)

In [29]:
X_train = train_processed
X_val = validation_processed

In [30]:
def minimal_features(df):
    df = df.copy()
    df['age'] = 2020 - df['year']
    df = df.drop(columns=["year"])
    df['mileage_per_year'] = df['mileage'] / (df['age'] + 1)
    df['efficiency_ratio'] = df['mpg'] / (df['engineSize'] + 0.1)
    df['age_mileage'] = df['age'] * df['mileage'] / 100000

    return df



In [31]:
X_train = minimal_features(X_train)
X_val = minimal_features(X_val)

In [32]:
model_mean_price = X_train.groupby('model')['price'].mean() 
X_train['model_mean_price'] = X_train['model'].map(model_mean_price) 
X_val['model_mean_price'] = X_val['model'].map(model_mean_price)

In [33]:
X_train = X_train.drop(columns=["model"])
X_val = X_val.drop(columns=["model"])

In [34]:
y_train = X_train['price']
y_val = X_val['price']

In [35]:
X_train = X_train.drop(columns=['price'])

X_val = X_val.drop(columns=['price'])


In [36]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60778 entries, 0 to 60777
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int32  
 1   tax               60778 non-null  int32  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   previousOwners    60778 non-null  int32  
 5   stated_no_damage  60778 non-null  float64
 6   Brand             60778 non-null  object 
 7   transmission      60778 non-null  object 
 8   fuelType          60778 non-null  object 
 9   age               60778 non-null  int32  
 10  mileage_per_year  60778 non-null  float64
 11  efficiency_ratio  60778 non-null  float64
 12  age_mileage       60778 non-null  float64
 13  model_mean_price  60778 non-null  float64
dtypes: float64(7), int32(4), object(3)
memory usage: 5.8+ MB


In [37]:
X_val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15195 entries, 0 to 15194
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           15195 non-null  int32  
 1   tax               15195 non-null  int32  
 2   mpg               15195 non-null  float64
 3   engineSize        15195 non-null  float64
 4   previousOwners    15195 non-null  int32  
 5   stated_no_damage  15195 non-null  float64
 6   Brand             15195 non-null  object 
 7   transmission      15195 non-null  object 
 8   fuelType          15195 non-null  object 
 9   age               15195 non-null  int32  
 10  mileage_per_year  15195 non-null  float64
 11  efficiency_ratio  15195 non-null  float64
 12  age_mileage       15195 non-null  float64
 13  model_mean_price  15195 non-null  float64
dtypes: float64(7), int32(4), object(3)
memory usage: 1.4+ MB


In [38]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 60778 entries, 0 to 60777
Series name: price
Non-Null Count  Dtype  
--------------  -----  
60778 non-null  float64
dtypes: float64(1)
memory usage: 712.2 KB


In [39]:
y_val.info()

<class 'pandas.core.series.Series'>
Index: 15195 entries, 0 to 15194
Series name: price
Non-Null Count  Dtype  
--------------  -----  
15195 non-null  float64
dtypes: float64(1)
memory usage: 178.1 KB


## Test of the models on the full dataset

In [40]:
class BrandModelTrainer:
    def __init__(self, estimator):
        self.estimator = estimator
        self.brand_models = {}
        self.feature_cols = None

    def fit(self, X_train, y_train):
        self.feature_cols = [c for c in X_train.columns if c != "Brand"]
        print(f"Training models for {len(X_train['Brand'].unique())} brands...\n")

        for brand in X_train["Brand"].unique():
            mask = X_train["Brand"] == brand
            Xb = X_train.loc[mask, self.feature_cols]
            yb = y_train[mask]

            numeric_cols = Xb.select_dtypes(include=["int64", "int32" ,"float64"]).columns.tolist()
            categorical_cols = Xb.select_dtypes(include=["object", "category"]).columns.tolist()

            preprocessor = ColumnTransformer([
                ("num", RobustScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
            ])

            model = Pipeline([
                ("preprocess", preprocessor),
                ("estimator", clone(self.estimator))
            ])

            model.fit(Xb, yb)
            self.brand_models[brand] = model
            print(f"  ✓ {brand} done.")

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for brand, model in self.brand_models.items():
            mask = X["Brand"] == brand
            if mask.sum() == 0:
                continue
            Xb = X.loc[mask, self.feature_cols]
            preds[mask] = model.predict(Xb)
        return preds

    def evaluate_train(self, X_train, y_train):
        y_pred = self.predict(X_train)
        rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        mae = mean_absolute_error(y_train, y_pred)
        r2 = r2_score(y_train, y_pred)
        print("\nTraining Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate(self, X_val, y_val):
        y_pred = self.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        print("\nValidation Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate_by_brand(self, X, y, split_name="Validation"):
        y_pred = self.predict(X)
        results = []
        for brand in X["Brand"].unique():
            mask = X["Brand"] == brand
            y_true_b = y[mask]
            y_pred_b = y_pred[mask]
            rmse = np.sqrt(mean_squared_error(y_true_b, y_pred_b))
            mae = mean_absolute_error(y_true_b, y_pred_b)
            r2 = r2_score(y_true_b, y_pred_b)
            results.append({"Brand": brand, "N": len(y_true_b), "RMSE": rmse, "MAE": mae, "R²": r2})
        df = pd.DataFrame(results).sort_values("RMSE")
        print(f"\n{split_name} Performance per Brand:")
        print(df.to_string(index=False))
        return df

    def evaluate_train_by_brand(self, X_train, y_train):
        return self.evaluate_by_brand(X_train, y_train, split_name="Training")
    
    def save_predictions(self, X, path):
        preds = self.predict(X)
        df = pd.DataFrame({
            "CarID": X["CarID"].values,
            "price": preds
        })
        df.to_csv(path, index=False)
        print(f"File salvato in: {path}")
        return df


In [41]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Linear Regression

In [ ]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 3442.01
  MAE:  2144.51
  R²:   0.8718

Validation Set Performance (Overall):
  RMSE: 3383.27
  MAE:  2156.67
  R²:   0.8723

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7644 1804.879210 1261.623779 0.740389
    ford 13108 2134.363308 1476.104103 0.800574
   skoda  3507 2189.201631 1541.424664 0.872451
  toyota  3775 2409.171888 1478.395462 0.854670
 hyundai  2724 2420.528081 1671.759526 0.835181
      vw  8479 3141.918645 2112.214045 0.832716
    audi  5971 4470.093313 2919.596232 0.855723
     bmw  6029 4489.217686 3085.250549 0.843073
mercedes  9541 5182.497736 3339.506559 0.763434

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1769.390434 1267.726325 0.753326
    ford 3282

,Brand,N,RMSE,MAE,R²
1,opel,1906,1769.390434,1267.726325,0.753326
2,ford,3282,2105.414326,1492.778850,0.801982
5,hyundai,683,2363.467323,1595.200160,0.846346
7,skoda,875,2510.576724,1588.635694,0.833582
3,toyota,941,2517.817091,1549.075444,0.837962
0,vw,2125,3337.402398,2147.630425,0.821398
6,audi,1489,4260.940846,2882.654055,0.848291
8,bmw,1513,4437.867434,3058.617525,0.849100
4,mercedes,2381,4919.273107,3374.256001,0.769401


### Decision Tree

In [ ]:
Tree = DecisionTreeRegressor(criterion="absolute_error")
trainer = BrandModelTrainer(Tree)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 112.79
  MAE:  6.37
  R²:   0.9999

Validation Set Performance (Overall):
  RMSE: 2784.00
  MAE:  1663.58
  R²:   0.9135

Training Performance per Brand:
   Brand     N       RMSE       MAE       R²
  toyota  3775  46.897353  2.598411 0.999945
      vw  8479  63.082838  3.946574 0.999933
    ford 13108  63.980359  3.652426 0.999821
   skoda  3507  80.171532  3.627887 0.999829
 hyundai  2724 104.155336  6.392070 0.999695
    audi  5971 118.844654  6.502261 0.999898
     bmw  6029 124.844385  3.790678 0.999879
    opel  7644 131.589032 11.239832 0.998620
mercedes  9541 181.644333 12.369982 0.999709

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1508.156131 1010.035705 0.820788
    ford 3282 1768.188805 1163.698578 0.860335


,Brand,N,RMSE,MAE,R²
1,opel,1906,1508.156131,1010.035705,0.820788
2,ford,3282,1768.188805,1163.698578,0.860335
5,hyundai,683,1912.997533,1211.926259,0.899336
3,toyota,941,2110.132421,1232.335501,0.886188
7,skoda,875,2189.804973,1472.373080,0.873391
0,vw,2125,2779.461077,1688.750188,0.876123
8,bmw,1513,3565.430794,2273.499693,0.902599
6,audi,1489,3731.609089,2353.809217,0.883643
4,mercedes,2381,3883.268842,2404.395680,0.856302


### KNR

In [ ]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 2086.84
  MAE:  1283.10
  R²:   0.9529

Validation Set Performance (Overall):
  RMSE: 2623.89
  MAE:  1599.60
  R²:   0.9232

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7644 1060.443151  724.541970 0.910381
    ford 13108 1306.958090  864.458222 0.925223
  toyota  3775 1415.382027  878.545362 0.949839
 hyundai  2724 1533.182629  989.739145 0.933874
   skoda  3507 1560.687760 1113.738857 0.935176
      vw  8479 1924.798390 1318.260710 0.937218
    audi  5971 2830.259754 1827.477616 0.942162
mercedes  9541 2885.203570 1869.137089 0.926679
     bmw  6029 2951.503736 1869.829258 0.932167

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1302.391145  882.899637 0.866353
    ford 3282

,Brand,N,RMSE,MAE,R²
1,opel,1906,1302.391145,882.899637,0.866353
2,ford,3282,1575.199951,1082.577625,0.889159
3,toyota,941,1745.839874,1107.878668,0.922093
5,hyundai,683,1856.141027,1194.873983,0.905231
7,skoda,875,1957.684444,1407.985499,0.898809
0,vw,2125,2484.154084,1624.600940,0.901048
8,bmw,1513,3484.453688,2329.667173,0.906973
4,mercedes,2381,3640.606197,2366.027305,0.873700
6,audi,1489,3810.683371,2262.526043,0.878660


### Random Forest

In [ ]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train, y_train)


rf_trainer.evaluate_train(X_train, y_train)
rf_trainer.evaluate(X_val, y_val)

rf_trainer.evaluate_train_by_brand(X_train, y_train)
rf_trainer.evaluate_by_brand(X_val, y_val)



Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 783.19
  MAE:  470.30
  R²:   0.9934

Validation Set Performance (Overall):
  RMSE: 2032.89
  MAE:  1258.34
  R²:   0.9539

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7644  445.237109 301.775315 0.984202
    ford 13108  530.727184 343.508418 0.987669
  toyota  3775  561.355323 336.224850 0.992110
 hyundai  2724  567.357718 363.538041 0.990945
   skoda  3507  597.478664 404.029184 0.990499
      vw  8479  689.289589 445.810446 0.991949
     bmw  6029 1029.470880 640.502578 0.991748
    audi  5971 1034.713743 641.635071 0.992270
mercedes  9541 1118.873712 694.385848 0.988974

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1149.310525  780.338976 0.895924
    ford 3282 1351.280315

,Brand,N,RMSE,MAE,R²
1,opel,1906,1149.310525,780.338976,0.895924
2,ford,3282,1351.280315,930.478081,0.918432
5,hyundai,683,1423.578040,936.074861,0.944255
3,toyota,941,1521.813102,934.312362,0.940804
7,skoda,875,1562.612019,1074.404938,0.935530
0,vw,2125,1895.158361,1228.334302,0.942408
6,audi,1489,2599.268918,1666.287440,0.943545
8,bmw,1513,2686.359438,1722.229650,0.944707
4,mercedes,2381,2896.389793,1857.880577,0.920059


### Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=69,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train, y_train)


mlp_trainer.evaluate_train(X_train, y_train)
mlp_trainer.evaluate(X_val, y_val)


mlp_trainer.evaluate_train_by_brand(X_train, y_train)
mlp_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 59852720.28436311
Validation score: -9.266880
Iteration 2, loss = 59007365.20543310
Validation score: -8.928061
Iteration 3, loss = 54676844.81815294
Validation score: -7.525840
Iteration 4, loss = 41738644.98614478
Validation score: -4.268555
Iteration 5, loss = 21796883.07524724
Validation score: -0.896603
Iteration 6, loss = 8471215.93549464
Validation score: 0.163387
Iteration 7, loss = 5143618.36199932
Validation score: 0.319848
Iteration 8, loss = 4062500.79534993
Validation score: 0.407811
Iteration 9, loss = 3428061.86028418
Validation score: 0.468079
Iteration 10, loss = 3014297.13070471
Validation score: 0.515664
Iteration 11, loss = 2654734.59489821
Validation score: 0.554798
Iteration 12, loss = 2405561.22326486
Validation score: 0.584354
Iteration 13, loss = 2217646.07471571
Validation score: 0.610275
Iteration 14, loss = 2074960.47227245
Validation score: 0.629207
Iteration 15, loss = 1963516.83492540
Validation score: 

,Brand,N,RMSE,MAE,R²
1,opel,1906,1185.958936,825.331619,0.889181
2,ford,3282,1457.492770,1026.689923,0.905105
5,hyundai,683,1498.572902,991.528424,0.938227
3,toyota,941,1584.843834,1000.246157,0.935799
7,skoda,875,1685.425548,1201.032753,0.924998
0,vw,2125,2325.160546,1491.889290,0.913309
6,audi,1489,2968.604285,2011.282473,0.926362
8,bmw,1513,3051.200355,2103.005976,0.928668
4,mercedes,2381,3367.450326,2239.211709,0.891942


### Performance

| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3383.27 | 2156.67 | 0.8723 |
| Tree | 2784.00 | 1663.58 | 0.9135 |
| KNR|2623.89 |1599.60 |0.9232 |
|RF | 2032.89 |1258.34 |0.9539 |
|NN | 2319.24 |1466.96 |0.9400 |


 

## Feature selection - Filter Methods

In [ ]:
df = X_train.join(y_train)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60778 entries, 0 to 60777
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int32  
 1   tax               60778 non-null  int32  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   previousOwners    60778 non-null  int32  
 5   stated_no_damage  60778 non-null  float64
 6   Brand             60778 non-null  object 
 7   transmission      60778 non-null  object 
 8   fuelType          60778 non-null  object 
 9   age               60778 non-null  int32  
 10  mileage_per_year  60778 non-null  float64
 11  efficiency_ratio  60778 non-null  float64
 12  age_mileage       60778 non-null  float64
 13  model_mean_price  60778 non-null  float64
 14  price             60778 non-null  float64
dtypes: float64(8), int32(4), object(3)
memory usage: 6.3+ MB


In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
y = df['price']

numerical_cols = df.select_dtypes(include=['int64', "int32",'float64']).columns

# Drop 'carID' and 'price'
num_cols = [col for col in numerical_cols 
            if "_transformed" not in col and col not in ['price', 'carID']]

print(num_cols)
print(categorical_cols)

['mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']
['Brand', 'transmission', 'fuelType']


In [ ]:
#FILTER METHOD


#ANOVA FUNCTION

def anova_for_categorical(df, y, categorical_cols):
    # Align indices between df and y
    common_idx = df.index.intersection(y.index)
    df_aligned = df.loc[common_idx]
    y_aligned = y.loc[common_idx]
    
    f_scores, p_values = [], []
    for col in df_aligned.columns:
        if col in categorical_cols:
            groups = [y_aligned[df_aligned[col] == cat] for cat in df_aligned[col].dropna().unique()]
            if len(groups) > 1 and all(len(g) > 1 for g in groups):
                f_stat, p_val = stats.f_oneway(*groups)
            else:
                f_stat, p_val = 0.0, 1.0
        else:
            if df_aligned[col].nunique() > 1:
                # Remove NaN values for correlation calculation
                valid_idx = df_aligned[col].notna() & y_aligned.notna()
                if valid_idx.sum() > 1:
                    corr = np.corrcoef(df_aligned.loc[valid_idx, col], y_aligned[valid_idx])[0, 1]
                    f_stat = corr**2 * valid_idx.sum()
                    p_val = 0.0
                else:
                    f_stat, p_val = 0.0, 1.0
            else:
                f_stat, p_val = 0.0, 1.0
        f_scores.append(f_stat)
        p_values.append(p_val)
    
    return np.array(f_scores), np.array(p_values)


def filter_method_selection(X_train, y_train, categorical_cols, num_cols,
                            top_k=None,
                            var_threshold=0.01,
                            corr_threshold=0.85):
    
    print("FILTER METHOD (Variance + Spearman Correlation + ANOVA)")
    
    # Ensure indices match
    common_idx = X_train.index.intersection(y_train.index)
    X_train = X_train.loc[common_idx]
    y_train = y_train.loc[common_idx]
    
    # Filter numerical columns that exist in X_train
    num_cols_in_X = [col for col in num_cols if col in X_train.columns]
    
    # Variance Threshold
    if num_cols_in_X:
        vt_selector = VarianceThreshold(threshold=var_threshold)
        X_num_vt = pd.DataFrame(
            vt_selector.fit_transform(X_train[num_cols_in_X]),
            columns=np.array(num_cols_in_X)[vt_selector.get_support()],
            index=X_train.index
        )
        print(f"Removed {len(num_cols_in_X) - X_num_vt.shape[1]} low-variance numeric features.")
    else:
        X_num_vt = pd.DataFrame(index=X_train.index)
    
    # Spearman Correlation
    if not X_num_vt.empty and X_num_vt.shape[1] > 1:
        corr_matrix = X_num_vt.corr(method='spearman').abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
        X_num_corr = X_num_vt.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} correlated numeric features (Spearman |corr| > {corr_threshold}).")
    else:
        X_num_corr = X_num_vt
    
    # Filter categorical columns that exist in X_train
    categorical_cols_in_X = [col for col in categorical_cols if col in X_train.columns]
    
    # Combine numeric + categorical
    X_filtered = pd.concat([X_num_corr, X_train[categorical_cols_in_X]], axis=1)
    
    # ANOVA F-test
    f_scores, f_pvalues = anova_for_categorical(X_filtered, y_train, categorical_cols_in_X)
    f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-10)
    
    results_df = pd.DataFrame({
        'feature': X_filtered.columns,
        'ANOVA_F': f_scores,
        'ANOVA_p_value': f_pvalues,
        'ANOVA_norm': f_norm,
        'type': ['categorical' if c in categorical_cols_in_X else 'numerical' for c in X_filtered.columns]
    }).sort_values('ANOVA_norm', ascending=False)
    
    if top_k is None:
        selected_features = X_filtered.columns.tolist()
    else:
        selected_features = X_filtered.columns[np.argsort(f_norm)[-top_k:]].tolist()
    
    print(f"\nFilter method selected {len(selected_features)} features")
    
    return selected_features, X_filtered[selected_features], results_df



In [ ]:
y = df["price"]

df = df.drop(columns=["price"])

In [ ]:

#X_train = df[num_cols]
#y_train = df['price']

# Filter method (Variance + Spearman + ANOVA)
# Define categorical and numerical columns first (if not already defined)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=['int64',"int32", 'float64']).columns.tolist()

# Filter method with all required parameters
selected_filter, X_train_filtered, filter_df = filter_method_selection(
    df, 
    y,
    categorical_cols=categorical_cols,  # Add this
    num_cols=num_cols,                  # Add this
    top_k=None, 
    var_threshold=0.01, 
    corr_threshold=0.85
)

# Print features selected
print("\nFeatures selected by Filter Method:")
for f in selected_filter:
    print(f)

# Display feature importance scores
print("\nTop 10 Features by ANOVA Score:")
print(filter_df.head(10))



FILTER METHOD (Variance + Spearman Correlation + ANOVA)
Removed 0 low-variance numeric features.
Removed 2 correlated numeric features (Spearman |corr| > 0.85).

Filter method selected 12 features

Features selected by Filter Method:
mileage
tax
mpg
engineSize
previousOwners
stated_no_damage
age
efficiency_ratio
model_mean_price
Brand
transmission
fuelType

Top 10 Features by ANOVA Score:
             feature       ANOVA_F  ANOVA_p_value  ANOVA_norm         type
8   model_mean_price  34427.971184            0.0    1.000000    numerical
3         engineSize  22861.690368            0.0    0.664044    numerical
6                age  14640.658332            0.0    0.425255    numerical
10      transmission  13153.933343            0.0    0.382071  categorical
0            mileage  10714.468578            0.0    0.311214    numerical
1                tax   5713.455939            0.0    0.165954    numerical
2                mpg   5027.571589            0.0    0.146032    numerical
9       

In [ ]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Test models on reduced DF

In [42]:
X_train_filter = X_train[["mileage", "tax", "mpg", "engineSize", "previousOwners", "stated_no_damage", "age", "efficiency_ratio", 
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_train_filter

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model_mean_price,fuelType
0,11899,124,51.4,1.4,2,1.0,4,34.266667,opel,manual,10488.218094,petrol
1,18500,145,48.7,1.4,4,1.0,2,32.466667,audi,semi-auto,22860.502532,petrol
2,10658,145,46.3,1.5,2,1.0,1,28.937500,mercedes,semi-auto,23609.185947,petrol
3,1500,145,43.5,0.0,4,1.0,0,435.000000,audi,manual,22522.319540,petrol
4,18495,98,64.1,2.1,4,1.0,3,29.136364,mercedes,automatic,23609.185947,diesel
...,...,...,...,...,...,...,...,...,...,...,...,...
60773,14867,150,55.4,1.4,3,1.0,2,36.933333,opel,manual,8338.330769,petrol
60774,20100,145,52.3,1.2,3,1.0,2,40.230769,toyota,manual,12456.520101,petrol
60775,10498,145,68.9,1.0,0,1.0,2,62.636364,toyota,manual,8046.197685,petrol
60776,17502,145,40.4,2.0,4,1.0,3,19.238095,audi,semi-auto,30135.793388,petrol


In [43]:
X_val_filter = X_val[["mileage", "tax", "mpg", "engineSize", "previousOwners", "stated_no_damage", "age", "efficiency_ratio", 
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_val_filter

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model_mean_price,fuelType
0,14774,145,53.3,2.0,0,1.0,1,25.380952,vw,manual,16802.884146,diesel
1,16375,150,47.9,1.6,4,1.0,4,28.176471,ford,manual,10252.815470,petrol
2,32746,265,37.2,3.0,3,0.0,4,12.000000,bmw,semi-auto,19665.944282,petrol
3,10164,145,68.9,1.5,3,1.0,1,43.062500,bmw,automatic,15836.032550,diesel
4,11407,145,80.7,1.5,2,1.0,1,50.437500,ford,manual,13400.396964,diesel
...,...,...,...,...,...,...,...,...,...,...,...,...
15190,10766,20,51.4,1.4,2,1.0,1,34.266667,opel,manual,10488.218094,petrol
15191,5000,145,48.7,2.0,1,1.0,0,23.190476,vw,semi-auto,16802.884146,diesel
15192,28056,145,60.1,2.0,0,1.0,3,28.619048,ford,manual,15784.977335,diesel
15193,25169,145,56.5,1.0,4,1.0,2,51.363636,ford,manual,10252.815470,petrol


### Linear Regression

In [ ]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 3569.83
  MAE:  2224.49
  R²:   0.8621

Validation Set Performance (Overall):
  RMSE: 3501.39
  MAE:  2241.68
  R²:   0.8632

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7644 1847.478671 1275.024377 0.727989
   skoda  3507 2209.115055 1540.806376 0.870120
    ford 13108 2235.318701 1573.277430 0.781262
  toyota  3775 2433.505700 1476.753655 0.851720
 hyundai  2724 2526.620730 1704.753502 0.820417
      vw  8479 3188.715394 2132.994825 0.827696
    audi  5971 4607.100051 2997.744466 0.846743
     bmw  6029 4805.254287 3314.721301 0.820201
mercedes  9541 5358.237831 3483.853978 0.747118

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1812.706179 1280.647362 0.741101
    ford 3282

,Brand,N,RMSE,MAE,R²
1,opel,1906,1812.706179,1280.647362,0.741101
2,ford,3282,2207.180214,1599.699316,0.782377
5,hyundai,683,2486.132214,1649.172777,0.829983
7,skoda,875,2531.178369,1585.820124,0.830839
3,toyota,941,2535.753276,1531.276127,0.835645
0,vw,2125,3396.538431,2171.385451,0.815013
6,audi,1489,4454.373544,3000.955775,0.834204
8,bmw,1513,4714.310051,3296.302181,0.829715
4,mercedes,2381,5039.620621,3505.430332,0.757980


### Single tree

In [ ]:
Tree = DecisionTreeRegressor(criterion="absolute_error")
trainer = BrandModelTrainer(Tree)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 114.96
  MAE:  6.59
  R²:   0.9999

Validation Set Performance (Overall):
  RMSE: 2759.39
  MAE:  1650.79
  R²:   0.9150

Training Performance per Brand:
   Brand     N       RMSE       MAE       R²
  toyota  3775  46.897353  2.598411 0.999945
      vw  8479  63.082838  3.946574 0.999933
    ford 13108  63.980359  3.652426 0.999821
   skoda  3507  80.171532  3.627887 0.999829
    audi  5971 118.844654  6.502261 0.999898
     bmw  6029 124.844385  3.790678 0.999879
    opel  7644 131.589032 11.239832 0.998620
 hyundai  2724 147.792520 11.420338 0.999386
mercedes  9541 181.644333 12.369982 0.999709

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1489.669080 1003.150186 0.825154
    ford 3282 1814.337280 1190.181697 0.852950


,Brand,N,RMSE,MAE,R²
1,opel,1906,1489.669080,1003.150186,0.825154
2,ford,3282,1814.337280,1190.181697,0.852950
5,hyundai,683,1902.116447,1235.952315,0.900478
7,skoda,875,2090.996628,1438.788439,0.884559
3,toyota,941,2346.387743,1266.682005,0.859276
0,vw,2125,2521.645041,1578.123835,0.898039
8,bmw,1513,3631.626580,2332.910419,0.898948
6,audi,1489,3687.525193,2288.208390,0.886376
4,mercedes,2381,3859.881075,2385.607905,0.858027


In [ ]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 2101.29
  MAE:  1281.10
  R²:   0.9522

Validation Set Performance (Overall):
  RMSE: 2624.75
  MAE:  1598.18
  R²:   0.9231

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7644 1061.803550  726.367342 0.910150
    ford 13108 1298.672984  853.786253 0.926168
  toyota  3775 1305.945217  829.741756 0.957296
   skoda  3507 1546.850965 1097.138994 0.936320
 hyundai  2724 1553.896147  992.036005 0.932075
      vw  8479 1857.952137 1266.352114 0.941503
    audi  5971 2882.967420 1852.861059 0.939987
mercedes  9541 2935.803235 1895.555914 0.924085
     bmw  6029 3023.021640 1915.804820 0.928840

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1340.656131  905.444789 0.858385
    ford 3282

,Brand,N,RMSE,MAE,R²
1,opel,1906,1340.656131,905.444789,0.858385
2,ford,3282,1541.808631,1065.807156,0.893808
3,toyota,941,1664.797525,1057.595262,0.929158
5,hyundai,683,1765.038156,1171.877665,0.914305
7,skoda,875,1916.159213,1375.449742,0.903057
0,vw,2125,2395.816536,1559.189324,0.907960
8,bmw,1513,3542.433394,2357.861489,0.903851
6,audi,1489,3716.919635,2316.399531,0.884557
4,mercedes,2381,3765.042995,2407.257809,0.864918


In [ ]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train_filter, y_train)


rf_trainer.evaluate_train(X_train_filter, y_train)
rf_trainer.evaluate(X_val_filter, y_val)

rf_trainer.evaluate_train_by_brand(X_train_filter, y_train)
rf_trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 787.09
  MAE:  470.69
  R²:   0.9933

Validation Set Performance (Overall):
  RMSE: 2036.38
  MAE:  1254.22
  R²:   0.9537

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7644  445.583387 300.987988 0.984177
    ford 13108  535.050811 345.635490 0.987468
  toyota  3775  564.412680 339.965707 0.992024
 hyundai  2724  570.561065 362.710890 0.990842
   skoda  3507  591.453614 404.982711 0.990690
      vw  8479  686.241571 447.794275 0.992020
     bmw  6029 1019.104211 634.184072 0.991913
    audi  5971 1020.306011 637.106042 0.992483
mercedes  9541 1149.001100 698.037248 0.988372

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1143.391435  780.680350 0.896993
    ford 3282 1347.582089

,Brand,N,RMSE,MAE,R²
1,opel,1906,1143.391435,780.680350,0.896993
2,ford,3282,1347.582089,931.183258,0.918878
5,hyundai,683,1439.731060,928.386067,0.942983
7,skoda,875,1564.294389,1067.217619,0.935391
3,toyota,941,1571.925232,947.383192,0.936841
0,vw,2125,1889.320722,1226.463762,0.942763
6,audi,1489,2630.209902,1669.746477,0.942193
8,bmw,1513,2670.874361,1694.002100,0.945343
4,mercedes,2381,2898.125343,1847.452824,0.919963


### Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=69,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train_filter, y_train)


mlp_trainer.evaluate_train(X_train_filter, y_train)
mlp_trainer.evaluate(X_val_filter, y_val)


mlp_trainer.evaluate_train_by_brand(X_train_filter, y_train)
mlp_trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

Iteration 1, loss = 60074429.15314120
Validation score: -8.812077
Iteration 2, loss = 59153427.26676983
Validation score: -8.429295
Iteration 3, loss = 54038065.92276022
Validation score: -6.897440
Iteration 4, loss = 38504717.10375964
Validation score: -3.973099
Iteration 5, loss = 16150524.08295236
Validation score: -1.474015
Iteration 6, loss = 5774947.23333462
Validation score: -0.430285
Iteration 7, loss = 3898101.21302131
Validation score: 0.003783
Iteration 8, loss = 3113215.03595616
Validation score: 0.137238
Iteration 9, loss = 2652479.89972620
Validation score: 0.221500
Iteration 10, loss = 2354027.43999203
Validation score: 0.290069
Iteration 11, loss = 2146626.74663658
Validation score: 0.349042
Iteration 12, loss = 1999515.69672397
Validation score: 0.393220
Iteration 13, loss = 1892011.74866987
Validation score: 0.442029
Iteration 14, loss = 1816187.10595373
Validation score: 0.465819
Iteration 15, loss = 1751985.64477006
Validation score:

,Brand,N,RMSE,MAE,R²
1,opel,1906,1265.193099,884.901378,0.873878
2,ford,3282,1463.474109,1025.101353,0.904325
5,hyundai,683,1468.102410,997.308031,0.940713
3,toyota,941,1550.406242,972.589193,0.938559
7,skoda,875,1704.502860,1213.497803,0.923290
0,vw,2125,2234.134239,1452.120753,0.919964
6,audi,1489,2920.003752,1962.489014,0.928753
4,mercedes,2381,3319.253478,2221.531215,0.895013
8,bmw,1513,3362.738278,2268.145322,0.913358


### Results

Performance on the full dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3383.27 | 2156.67 | 0.8723 |
| Tree | 2784.00 | 1663.58 | 0.9135 |
| KNR|2623.89 |1599.60 |0.9232 |
|RF | 2032.89 |1258.34 |0.9539 |
|NN | 2319.24 |1466.96 |0.9400 |



Performance on the reduced dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3501.39 | 2241.68 | 0.8632 |
| Tree       | 2759.39 | 1650.79 | 0.9150 |
| KNR               | 2624.75 | 1598.18 | 0.9231 |
| RF                | **2036.38** | **1254.22** | **0.9537** |
| NN                | 2337.17.41 | 1476.69 | 0.9391 |



In [ ]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Feature Selection - Wrapper Method

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(num_cols)

['mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']


In [ ]:
def rfe(X_train, X_val, y_train, y_val, num_cols, step=1, n_estimators=100, random_state=69):
    
    valid_num_cols = [col for col in num_cols if col in X_train.columns]
    if len(valid_num_cols) == 0:
        raise ValueError("No valid numeric columns found in X_train.")
    print(f"Using {len(valid_num_cols)} numeric columns for RFE:\n{valid_num_cols}")
    
    X_train_num = X_train[valid_num_cols]
    X_val_num = X_val[valid_num_cols]
    nof_list = np.arange(1, X_train_num.shape[1]+1)
    
    best_mae = float('inf')
    nof = 0
    train_mae_list = []
    val_mae_list = []
    features_to_select = None
    
    for n in nof_list:
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
        rfe = RFE(estimator=model, n_features_to_select=n, step=step)
        X_train_rfe = rfe.fit_transform(X_train_num, y_train)
        X_val_rfe = rfe.transform(X_val_num)
        
        model.fit(X_train_rfe, y_train)
        
        train_pred = model.predict(X_train_rfe)
        val_pred = model.predict(X_val_rfe)
        train_mae = mean_absolute_error(y_train, train_pred)
        val_mae = mean_absolute_error(y_val, val_pred)
        
        train_mae_list.append(train_mae)
        val_mae_list.append(val_mae)
        
        if val_mae < best_mae:
            best_mae = val_mae
            nof = n
            features_to_select = pd.Series(rfe.support_, index=X_train_num.columns)
    
    selected_features = features_to_select[features_to_select].index.tolist()
    
    print(f"\nOptimum number of features: {nof}")
    print(f"Best validation MAE: {best_mae:.4f}")
    print("Selected features:")
    print(selected_features)
    
    return nof, best_mae, selected_features, train_mae_list, val_mae_list

In [ ]:
nof, best_score, selected_rfe_features, train_scores, val_scores = rfe(
    X_train, X_val, y_train, y_val, num_cols
)

Using 11 numeric columns for RFE:
['mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']

Optimum number of features: 9
Best validation MAE: 1323.6319
Selected features:
['mileage', 'tax', 'mpg', 'engineSize', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'model_mean_price']


In [ ]:
X_train_rfed = X_train[["mileage", "tax", "mpg", "engineSize", "age", "efficiency_ratio", "mileage_per_year", "age_mileage",
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_train_rfed

,mileage,tax,mpg,engineSize,age,efficiency_ratio,mileage_per_year,age_mileage,Brand,transmission,model_mean_price,fuelType
0,16326,125,51.4,1.4,3,34.266667,4081.500000,0.48978,opel,manual,10496.007929,petrol
1,12510,145,56.5,1.0,2,51.363636,4170.000000,0.25020,toyota,manual,8029.315503,petrol
2,6999,144,58.2,1.0,1,52.909091,3499.500000,0.06999,ford,manual,13441.487966,petrol
3,28465,160,43.5,2.0,3,20.714286,7116.250000,0.85395,mercedes,manual,30801.319510,petrol
4,9221,145,33.6,2.0,1,16.000000,4610.500000,0.09221,vw,semi-auto,34233.178082,electric
...,...,...,...,...,...,...,...,...,...,...,...,...
60773,21131,145,67.3,2.1,1,30.590909,10565.500000,0.21131,mercedes,semi-auto,20327.302521,diesel
60774,4722,145,50.4,1.6,1,29.647059,2361.000000,0.04722,vw,manual,22619.961912,diesel
60775,24755,0,76.3,1.6,5,44.882353,4125.833333,1.23775,audi,manual,14343.507937,diesel
60776,42000,125,57.6,2.0,5,27.428571,7000.000000,2.10000,bmw,manual,15717.953402,diesel


In [ ]:
X_val_rfed = X_val[["mileage", "tax", "mpg", "engineSize", "age", "efficiency_ratio", "mileage_per_year", "age_mileage",
                          "Brand", "transmission", "model_mean_price", "fuelType"]]
X_val_rfed

,mileage,tax,mpg,engineSize,age,efficiency_ratio,mileage_per_year,age_mileage,Brand,transmission,model_mean_price,fuelType
0,5000,145,48.7,2.0,1,23.190476,2500.000000,0.05000,vw,automatic,17101.053465,diesel
1,10,145,43.5,1.4,1,29.000000,5.000000,0.00010,opel,manual,8348.382147,petrol
2,46000,125,51.4,1.0,4,46.727273,9200.000000,1.84000,ford,semi-auto,13441.487966,petrol
3,24219,20,60.1,1.2,3,46.230769,6054.750000,0.72657,vw,manual,11435.947368,petrol
4,20251,145,46.3,1.8,4,24.368421,4050.200000,0.81004,toyota,automatic,10373.418919,petrol
...,...,...,...,...,...,...,...,...,...,...,...,...
15190,33154,30,54.3,1.2,6,41.769231,4736.285714,1.98924,ford,manual,10245.037681,petrol
15191,19000,20,60.1,1.0,4,54.636364,3800.000000,0.76000,vw,manual,11435.947368,petrol
15192,3971,145,45.6,1.5,1,28.500000,1985.500000,0.03971,vw,manual,16722.731829,petrol
15193,56603,0,83.1,1.4,4,55.400000,11320.600000,2.26412,vw,manual,11435.947368,diesel


In [ ]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train_rfed, y_train)


rf_trainer.evaluate_train(X_train_rfed, y_train)
rf_trainer.evaluate(X_val_rfed, y_val)

rf_trainer.evaluate_train_by_brand(X_train_rfed, y_train)
rf_trainer.evaluate_by_brand(X_val_rfed, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 791.34
  MAE:  473.66
  R²:   0.9932

Validation Set Performance (Overall):
  RMSE: 2033.05
  MAE:  1256.58
  R²:   0.9539

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7644  463.914285 309.235998 0.982848
    ford 13108  535.870395 345.862177 0.987429
  toyota  3775  561.716833 338.077681 0.992100
 hyundai  2724  577.270768 367.348817 0.990626
   skoda  3507  602.826087 409.353137 0.990329
      vw  8479  708.249640 454.007482 0.991500
    audi  5971 1037.789691 645.645367 0.992224
     bmw  6029 1039.159002 641.595021 0.991591
mercedes  9541 1125.371000 692.305729 0.988845

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1151.895521  786.584062 0.895455
    ford 3282 1354.041743

,Brand,N,RMSE,MAE,R²
1,opel,1906,1151.895521,786.584062,0.895455
2,ford,3282,1354.041743,935.275800,0.918098
5,hyundai,683,1438.914752,944.670402,0.943047
7,skoda,875,1554.184116,1073.572999,0.936224
3,toyota,941,1560.618769,952.116895,0.937747
0,vw,2125,1904.666983,1228.593323,0.941829
6,audi,1489,2602.379454,1662.530539,0.943410
8,bmw,1513,2680.604426,1713.106259,0.944944
4,mercedes,2381,2881.876592,1833.798991,0.920858


Final Results

Performance on the full dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3383.27 | 2156.67 | 0.8723 |
| Tree | 2784.00 | 1663.58 | 0.9135 |
| KNR|2623.89 |1599.60 |0.9232 |
|RF | 2032.89 |1258.34 |0.9539 |
|NN | 2319.24 |1466.96 |0.9400 |



Performance on the reduced dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3501.39 | 2241.68 | 0.8632 |
| Tree       | 2759.39 | 1650.79 | 0.9150 |
| KNR               | 2624.75 | 1598.18 | 0.9231 |
| RF                | **2036.38** | **1254.22** | **0.9537** |
| NN                | 2337.17.41 | 1476.69 | 0.9391 |


Random Forest on the rfed dataset

|*Model*| *RMSE*| *MAE* | *R²*|
|----|----|----|----|
|RF| 2033.05| 1256.58 | 0.9539|

## Hyperparam tuning

In [ ]:
class HoldoutRandomSearch:
    def __init__(self, trainer_class, param_space, n_iter=20, optimize_metric='mae'):
        self.trainer_class = trainer_class
        self.param_space = param_space
        self.n_iter = n_iter
        self.optimize_metric = optimize_metric.lower()
        self.results = []
        self.best_params = None
        self.best_score = None
        self.best_trainer = None
        self.brand_best_configs = {}
        
    def sample_params(self):
        if isinstance(self.param_space, list):
            return random.choice(self.param_space)
        else:
            return {k: random.choice(v) for k, v in self.param_space.items()}
    
    def _get_param_summary(self, params):
        summary = {}
        
        if 'n_estimators' in params:
            summary['n_estimators'] = params['n_estimators']
        if 'max_depth' in params:
            summary['max_depth'] = params['max_depth']
        if 'max_features' in params:
            summary['max_features'] = params['max_features']
            
        if 'hidden_layer_sizes' in params:
            summary['hidden_layers'] = str(params['hidden_layer_sizes'])
        if 'alpha' in params:
            summary['alpha'] = params['alpha']
        if 'learning_rate_init' in params:
            summary['lr'] = params['learning_rate_init']
        if 'activation' in params:
            summary['activation'] = params['activation']
            
        return summary
    
    def run(self, X_train, y_train, X_val, y_val):
        metric_name = self.optimize_metric.upper()
        print(f"Running random search ({self.n_iter} iterations)...")
        print(f"Optimizing for: {metric_name}\n")
        
        start_time = time.time()
        brands = X_train["Brand"].unique()

        for brand in brands:
            self.brand_best_configs[brand] = {
                'score': float('inf'),
                'params': None,
                'config_num': None,
                'all_metrics': {}
            }
        
        for i in range(self.n_iter):
            iter_start = time.time()
            params = self.sample_params()
            
            print(f"\n{'='*70}")
            print(f"[{i+1}/{self.n_iter}] Testing params:")
            print(params)
            print('='*70)
            

            estimator = self.trainer_class.estimator.__class__(**params)
            trainer = BrandModelTrainer(estimator)
            trainer.fit(X_train, y_train)

            metrics = trainer.evaluate(X_val, y_val)
            rmse = metrics["RMSE"]
            mae = metrics["MAE"]
            r2 = metrics["R²"]
            
            current_score = mae if self.optimize_metric == 'mae' else rmse
            
            brand_results = {}
            for brand in brands:
                mask = X_val["Brand"] == brand
                y_true_brand = y_val[mask]
                y_pred_brand = trainer.predict(X_val[mask])
                
                brand_rmse = np.sqrt(mean_squared_error(y_true_brand, y_pred_brand))
                brand_mae = mean_absolute_error(y_true_brand, y_pred_brand)
                brand_r2 = r2_score(y_true_brand, y_pred_brand)
                
                brand_score = brand_mae if self.optimize_metric == 'mae' else brand_rmse
                
                brand_results[brand] = {
                    'score': brand_score,
                    'rmse': brand_rmse,
                    'mae': brand_mae,
                    'r2': brand_r2
                }
                
                if brand_score < self.brand_best_configs[brand]['score']:
                    self.brand_best_configs[brand]['score'] = brand_score
                    self.brand_best_configs[brand]['params'] = params.copy()
                    self.brand_best_configs[brand]['config_num'] = i + 1
                    self.brand_best_configs[brand]['all_metrics'] = {
                        'rmse': brand_rmse,
                        'mae': brand_mae,
                        'r2': brand_r2
                    }
            
            self.results.append({
                "config_num": i + 1,
                "params": params,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "score": current_score,
                "brand_results": brand_results
            })

            if self.best_score is None or current_score < self.best_score:
                self.best_score = current_score
                self.best_params = params
                self.best_trainer = trainer
                print(f"New best overall model ({metric_name}: {current_score:.2f})")
           

            elapsed_total = time.time() - start_time
            avg_per_iter = elapsed_total / (i + 1)
            eta = avg_per_iter * (self.n_iter - (i + 1))
            
            print(f"\nProgress: {i+1}/{self.n_iter} | Elapsed: {elapsed_total:.1f}s | ETA: ~{eta:.1f}s")
        
        print(f"SEARCH COMPLETED - FINAL RESULTS (Optimized for {metric_name})")
        
        print(f"\n Best general model:")
        print(f"  Best {metric_name}: {self.best_score:.2f}")
        
        best_result = [r for r in self.results if r['score'] == self.best_score][0]
        print(f"  RMSE: {best_result['rmse']:.2f}")
        print(f"  MAE:  {best_result['mae']:.2f}")
        print(f"  R²:   {best_result['r2']:.4f}")
        print(f"  Params: {self.best_params}")
   
        print(f"Best configuration per brand (by {metric_name})")

        
        brand_summary = []
        for brand in sorted(brands):
            config = self.brand_best_configs[brand]
            
            summary = {
                'Brand': brand,
                f'Best_{metric_name}': config['score'],
                'RMSE': config['all_metrics']['rmse'],
                'MAE': config['all_metrics']['mae'],
                'R²': config['all_metrics']['r2'],
                'Config_Num': config['config_num']
            }
            
            param_summary = self._get_param_summary(config['params'])
            summary.update(param_summary)
            
            brand_summary.append(summary)
            
            print(f"\n{brand.upper()}:")
            print(f"  Best {metric_name}: {config['score']:.2f}")
            print(f"  RMSE: {config['all_metrics']['rmse']:.2f}")
            print(f"  MAE:  {config['all_metrics']['mae']:.2f}")
            print(f"  R²:   {config['all_metrics']['r2']:.4f}")
            print(f"  Found at iteration: {config['config_num']}")
            print(f"  Best params:")
            for k, v in list(config['params'].items())[:5]:
                if k not in ['random_state', 'n_jobs', 'shuffle', 'verbose', 'warm_start']:
                    print(f"    {k}: {v}")
        
        df_summary = pd.DataFrame(brand_summary).sort_values(f'Best_{metric_name}')
  
        print(df_summary.to_string(index=False))
        
        results_df = pd.DataFrame([
            {
                'Config': r['config_num'],
                metric_name: r['score'],
                'RMSE': r['rmse'],
                'MAE': r['mae'],
                'R²': r['r2']
            }
            for r in self.results
        ]).sort_values(metric_name)
        
        print(f"ALL CONFIGURATIONS (sorted by {metric_name}):")
        print(results_df.head(10).to_string(index=False))
        
        return self.best_trainer, self.best_params, self.best_score

### Random Forest

In [ ]:
base_estimator = RandomForestRegressor(random_state=69, n_jobs=-1)
trainer = BrandModelTrainer(base_estimator)

param_space_rf = {
    "n_estimators": [400, 800, 1200],
    "max_depth": [None, 15, 25, 35],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.3, 0.5, 0.7, 1.0],
    "min_impurity_decrease": [0.0, 1e-4, 1e-3, 1e-2],
    "max_samples": [0.5, 0.7, 0.9, None],
    "criterion": ["squared_error", "friedman_mse"],
    "bootstrap": [True],
    "random_state": [69],
    "n_jobs": [-1]
}

search = HoldoutRandomSearch(
    trainer_class=trainer,
    param_space=param_space_rf,
    n_iter=60
)


best_trainer, best_params, best_rmse = search.run(X_train_filter, y_train, X_val_filter, y_val)


Running random search (60 iterations)...
Optimizing for: MAE


[1/60] Testing params:
{'n_estimators': 1200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.3, 'min_impurity_decrease': 0.0001, 'max_samples': 0.9, 'criterion': 'friedman_mse', 'bootstrap': True, 'random_state': 69, 'n_jobs': -1}
Training models for 9 brands...

  ✓ opel done.
  ✓ toyota done.
  ✓ ford done.
  ✓ mercedes done.
  ✓ vw done.
  ✓ bmw done.
  ✓ audi done.
  ✓ hyundai done.
  ✓ skoda done.

Validation Set Performance (Overall):
  RMSE: 2010.32
  MAE:  1245.43
  R²:   0.9549
New best overall model (MAE: 1245.43)

Progress: 1/60 | Elapsed: 84.7s | ETA: ~4996.7s

[2/60] Testing params:
{'n_estimators': 1200, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'min_impurity_decrease': 0.001, 'max_samples': 0.5, 'criterion': 'squared_error', 'bootstrap': True, 'random_state': 69, 'n_jobs': -1}
Training models for 9 brands...

  ✓ opel done.
  ✓ to

In [ ]:
best_trainer.evaluate_by_brand(X_val_filter, y_val)



Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1906 1086.769229  748.110206 0.906943
    ford 3282 1277.274084  901.254733 0.927122
 hyundai  683 1368.252148  890.840175 0.948504
  toyota  941 1378.231055  858.386788 0.951447
   skoda  875 1494.197535 1037.671806 0.941052
      vw 2125 1875.588743 1188.962644 0.943592
     bmw 1513 2457.630204 1624.097258 0.953722
    audi 1489 2543.390249 1621.939526 0.945946
mercedes 2381 2793.058290 1790.165923 0.925661


,Brand,N,RMSE,MAE,R²
1,opel,1906,1086.769229,748.110206,0.906943
2,ford,3282,1277.274084,901.254733,0.927122
5,hyundai,683,1368.252148,890.840175,0.948504
3,toyota,941,1378.231055,858.386788,0.951447
7,skoda,875,1494.197535,1037.671806,0.941052
0,vw,2125,1875.588743,1188.962644,0.943592
8,bmw,1513,2457.630204,1624.097258,0.953722
6,audi,1489,2543.390249,1621.939526,0.945946
4,mercedes,2381,2793.058290,1790.165923,0.925661


In [ ]:
raise SystemExit("Stop before training the models")

## Predictions

In [44]:
best_params_per_brand = {
    'opel': {
        'n_estimators': 1200,
        'max_depth': 15,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.7,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'toyota': {
        'n_estimators': 800,
        'max_depth': 25,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.3,
        'min_impurity_decrease': 0.01,
        'max_samples': None,
        'criterion': 'friedman_mse',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'ford': {
        'n_estimators': 1200,
        'max_depth': 15,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.7,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'hyundai': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'skoda': {
        'n_estimators': 1200,
        'max_depth': 15,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.7,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'vw': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'bmw': {
        'n_estimators': 400,
        'max_depth': 35,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'audi': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    },
    'mercedes': {
        'n_estimators': 1200,
        'max_depth': None,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.5,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'criterion': 'squared_error',
        'bootstrap': True,
        'random_state': 69,
        'n_jobs': -1
    }
}

In [49]:
X_full_train = pd.concat([X_train_filter, X_val_filter], axis=0)
y_full_train = pd.concat([y_train, y_val], axis=0)

numeric_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'age', 
                 'efficiency_ratio', 
                'model_mean_price']

categorical_cols = ['Brand', 'transmission', 'fuelType']


In [50]:
models = {}
scalers = {}
brand_encoders = {}


brands = X_full_train['Brand'].unique()

for brand in brands:
    brand_lower = brand.lower()
    print(f"\n Training model for {brand.upper()}")
    
    mask_train = X_full_train['Brand'] == brand
    X_brand_train = X_full_train[mask_train].copy()
    y_brand_train = y_full_train[mask_train].copy()
    
    X_numeric = X_brand_train[numeric_cols].copy()
    
    scaler = RobustScaler()
    X_numeric_scaled = scaler.fit_transform(X_numeric)
    X_numeric_scaled = pd.DataFrame(
        X_numeric_scaled, 
        columns=numeric_cols,
        index=X_brand_train.index
    )

    scalers[brand] = scaler

    X_brand_encoded = X_numeric_scaled.copy()
    
    for col in ['transmission', 'fuelType']:
        dummies = pd.get_dummies(X_brand_train[col], prefix=col, drop_first=True)
        X_brand_encoded = pd.concat([X_brand_encoded, dummies], axis=1)
    

    brand_encoders[brand] = {
        'columns': X_brand_encoded.columns.tolist(),
        'numeric_cols': numeric_cols
    }
    
    if brand_lower in best_params_per_brand:
        params = best_params_per_brand[brand_lower]
    else:
        params = {
            'n_estimators': 1200,
            'max_depth': None,
            'min_samples_split': 5,
            'min_samples_leaf': 1,
            'max_features': 0.5,
            'random_state': 69,
            'n_jobs': -1
        }
    
    model = RandomForestRegressor(**params)
    model.fit(X_brand_encoded, y_brand_train)
  
    models[brand] = model
    
    print(f" {brand.upper()} model trained")


 Training model for OPEL
 OPEL model trained

 Training model for AUDI
 AUDI model trained

 Training model for MERCEDES
 MERCEDES model trained

 Training model for FORD
 FORD model trained

 Training model for BMW
 BMW model trained

 Training model for TOYOTA
 TOYOTA model trained

 Training model for HYUNDAI
 HYUNDAI model trained

 Training model for VW
 VW model trained

 Training model for SKODA
 SKODA model trained


In [54]:
test_processed.head()

,year,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,Brand,transmission,model,fuelType
carID,,,,,,,,,,,
89856,2015,30700,205,41.5,1.6,3,1.0,hyundai,automatic,i30,petrol
106581,2017,48191,150,38.2,2.0,2,1.0,vw,semi-auto,tiguan,petrol
80886,2016,36792,125,51.4,1.5,2,1.0,bmw,automatic,2 series,petrol
100174,2019,5533,145,44.1,1.2,1,1.0,opel,manual,grandland x,petrol
81376,2019,9058,150,51.4,2.0,4,1.0,bmw,semi-auto,1 series,diesel


In [55]:
X_test = minimal_features(test_processed)
X_test

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,Brand,transmission,model,fuelType,age,mileage_per_year,efficiency_ratio,age_mileage
carID,,,,,,,,,,,,,,
89856,30700,205,41.5,1.6,3,1.0,hyundai,automatic,i30,petrol,5,5116.666667,24.411765,1.53500
106581,48191,150,38.2,2.0,2,1.0,vw,semi-auto,tiguan,petrol,3,12047.750000,18.190476,1.44573
80886,36792,125,51.4,1.5,2,1.0,bmw,automatic,2 series,petrol,4,7358.400000,32.125000,1.47168
100174,5533,145,44.1,1.2,1,1.0,opel,manual,grandland x,petrol,1,2766.500000,33.923077,0.05533
81376,9058,150,51.4,2.0,4,1.0,bmw,semi-auto,1 series,diesel,1,4529.000000,24.476190,0.09058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105775,27575,145,46.3,1.4,1,1.0,vw,manual,tiguan,petrol,3,6893.750000,30.866667,0.82725
81363,1980,145,34.0,2.0,3,1.0,bmw,automatic,x2,petrol,0,1980.000000,16.190476,0.00000
76833,8297,145,38.2,2.0,4,1.0,audi,semi-auto,q5,diesel,1,4148.500000,18.190476,0.08297


In [58]:
X_test['model_mean_price'] = X_test['model'].map(model_mean_price)

In [61]:
X_train_filter.head()

,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model_mean_price,fuelType
0,11899,124,51.4,1.4,2,1.0,4,34.266667,opel,manual,10488.218094,petrol
1,18500,145,48.7,1.4,4,1.0,2,32.466667,audi,semi-auto,22860.502532,petrol
2,10658,145,46.3,1.5,2,1.0,1,28.937500,mercedes,semi-auto,23609.185947,petrol
3,1500,145,43.5,0.0,4,1.0,0,435.000000,audi,manual,22522.319540,petrol
4,18495,98,64.1,2.1,4,1.0,3,29.136364,mercedes,automatic,23609.185947,diesel


In [62]:
X_test = X_test.drop(columns=["model", "mileage_per_year", "age_mileage"])
X_test.head()


,mileage,tax,mpg,engineSize,previousOwners,stated_no_damage,Brand,transmission,fuelType,age,efficiency_ratio,model_mean_price
carID,,,,,,,,,,,,
89856,30700,205,41.5,1.6,3,1.0,hyundai,automatic,petrol,5,24.411765,11848.081505
106581,48191,150,38.2,2.0,2,1.0,vw,semi-auto,petrol,3,18.190476,21509.106996
80886,36792,125,51.4,1.5,2,1.0,bmw,automatic,petrol,4,32.125000,19665.944282
100174,5533,145,44.1,1.2,1,1.0,opel,manual,petrol,1,33.923077,17217.228906
81376,9058,150,51.4,2.0,4,1.0,bmw,semi-auto,diesel,1,24.476190,15836.032550


In [63]:
predictions = np.zeros(len(X_test))

for brand in X_test['Brand'].unique():
    print(f"\n Predicting for {brand.upper()}...")

    mask_test = X_test['Brand'] == brand
    X_brand_test = X_test[mask_test].copy()
    
    X_numeric_test = X_brand_test[numeric_cols].copy()
    
    scaler = scalers[brand]
    X_numeric_test_scaled = scaler.transform(X_numeric_test)
    X_numeric_test_scaled = pd.DataFrame(
        X_numeric_test_scaled,
        columns=numeric_cols,
        index=X_brand_test.index
    )
    
    X_brand_test_encoded = X_numeric_test_scaled.copy()
    
    for col in ['transmission', 'fuelType']:
        dummies = pd.get_dummies(X_brand_test[col], prefix=col, drop_first=True)
        X_brand_test_encoded = pd.concat([X_brand_test_encoded, dummies], axis=1)

    expected_cols = brand_encoders[brand]['columns']

    for col in expected_cols:
        if col not in X_brand_test_encoded.columns:
            X_brand_test_encoded[col] = 0
    
    X_brand_test_encoded = X_brand_test_encoded[expected_cols]
    
    model = models[brand]
    brand_predictions = model.predict(X_brand_test_encoded)

    predictions[mask_test] = brand_predictions
    



 Predicting for HYUNDAI...

 Predicting for VW...

 Predicting for BMW...

 Predicting for OPEL...

 Predicting for FORD...

 Predicting for MERCEDES...

 Predicting for SKODA...

 Predicting for TOYOTA...

 Predicting for AUDI...


In [64]:
if X_test.index.name == 'carID' or 'carID' in X_test.index.names:
    car_ids = X_test.index
else:
    car_ids = X_test.index

submission = pd.DataFrame({
    'carID': car_ids,
    'price': predictions
})

submission = submission.sort_values('carID')


submission['price'] = submission['price'].round(2)

output_filename = 'submission.csv'
submission.to_csv(output_filename, index=False)
